In [70]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI

In [71]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph = StateGraph(State)

In [72]:
%pip install -U langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [73]:
@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["yorkshire"]:
        return "It's cold and wet."
    else:
        return "It's warm and sunny."
    
import os
from langchain_groq import ChatGroq
# Load GROQ_API_KEY from environment or .env file
if 'GROQ_API_KEY' not in os.environ and os.path.exists('.env'):
    with open('.env', 'r', encoding='utf-8') as f:
        for env_line in f:
            if env_line.strip().startswith('GROQ_API_KEY='):
                os.environ['GROQ_API_KEY'] = env_line.strip().split('=', 1)[1].strip("'\"")
llm = ChatGroq(model="openai/gpt-oss-120b")


tools = [get_weather]

llm_with_tools = llm.bind_tools(tools)

tool_node = ToolNode(tools)

graph.add_node("tool_node", tool_node)

In [74]:
from langgraph.graph import StateGraph

# Create a fresh graph
graph = StateGraph(State)

def prompt_node(state: State) -> dict:
    new_message = llm_with_tools.invoke(state["messages"])
    return {"messages": [new_message]}

graph.add_node("prompt_node", prompt_node)
graph.add_node("tool_node", tool_node) 

In [75]:
def conditional_edge(state: State) -> Literal['tool_node', '__end__']:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    else:
        return "__end__"
    
graph.add_conditional_edges(
    'prompt_node',
    conditional_edge
)
graph.add_edge("tool_node", "prompt_node")
graph.set_entry_point("prompt_node")


In [ ]:
APP = graph.compile()

print("=" * 60)
print("Assistant is ready! Type your question below.")
print("Type 'exit' or 'quit' to stop.")
print("=" * 60 + "\n")

chat_history = []

while True:
    user_question = input("Enter your question: ").strip()
    
    if user_question.lower() in ["exit", "quit", "q"]:
        print("\nExiting chat. Goodbye!")
        break
        
    if not user_question:
        continue

    # Append question and invoke LangGraph
    chat_history.append(("user", user_question))
    response = APP.invoke({"messages": chat_history})
    
    # Keep full conversation history so follow-ups work seamlessly
    chat_history = response["messages"]
    
    print("\nAnswer:")
    print(chat_history[-1].content)
    print("-" * 60 + "\n")


Assistant is ready! Type your question below.
Type 'exit' or 'quit' to stop.


Answer:
The current weather in Windsor, UK is warm and sunny.
------------------------------------------------------------


Answer:
The weather in Delhi today is warm and sunny.
------------------------------------------------------------

